# Three Uses of the MoveOn Public Listings API

This notebook demonstrates three different purposes and outputs using the public `/api/listings/` endpoint:

1. **Spreadsheet reporting:** export listings and category price summaries to an Excel workbook.
2. **Interactive data exploration:** compare prices and category counts with interactive charts.
3. **A small modeling experiment:** predict listing prices from category and condition, compared against a median-price baseline.

## Setup and API data

From the repository root, install the project and notebook dependencies with `pip install -r requirements-notebook.txt`. This includes the web-app requirements. Apply migrations and seed active listings in the development database, then start the Django server with `python manage.py runserver 127.0.0.1:8000 --settings=moveon.settings.development`. Run this notebook with that same Python environment. Change the API base URL below if the server uses another host or port.

In [1]:
from pathlib import Path

import pandas as pd
import requests

API_BASE_URL = "http://127.0.0.1:8000/api/listings/"
TIMEOUT_SECONDS = 10

def fetch_all_listings(base_url=API_BASE_URL):
    """Fetch every page from the public listings endpoint."""
    first = requests.get(base_url, timeout=TIMEOUT_SECONDS)
    first.raise_for_status()
    payload = first.json()
    rows = list(payload.get("results", []))
    for page_number in range(2, payload.get("pages", 1) + 1):
        response = requests.get(base_url, params={"page": page_number}, timeout=TIMEOUT_SECONDS)
        response.raise_for_status()
        rows.extend(response.json().get("results", []))
    if not rows:
        raise ValueError("The API returned no listings. Seed active listings and check the server/database.")
    frame = pd.DataFrame(rows)
    frame["category"] = frame["category"].map(lambda value: value.get("name", "Unknown") if isinstance(value, dict) else "Unknown")
    frame["price"] = pd.to_numeric(frame["price"], errors="coerce")
    frame["condition"] = frame["condition"].fillna("Unknown")
    return frame.dropna(subset=["price"]).reset_index(drop=True)

listings = fetch_all_listings()
print(f"API: {API_BASE_URL}")
print(f"Rows fetched across all pages: {len(listings)}")
display(listings[["id", "title", "price", "condition", "category"]].head())

API: http://127.0.0.1:8000/api/listings/
Rows fetched across all pages: 42


,id,title,price,condition,category
0,43,Bookshelf,45.0,GOOD,Furniture
1,42,Desk Lamp,22.0,GOOD,Home Decor
2,41,Oak Desk,60.0,GOOD,Furniture
3,40,Throw Pillow,12.0,GOOD,Home Decor
4,39,Coffee Table,40.0,GOOD,Furniture


## Use 1: Excel price summary

**Purpose:** create a shareable spreadsheet for comparing prices across categories and conditions.

**Output:** an `.xlsx` workbook with listing rows and category/condition summaries. It is written to `data/`, which is excluded from Git because it contains generated local data.

In [2]:
category_summary = (
    listings.groupby("category")["price"]
    .agg(listings="count", average_price="mean", median_price="median", min_price="min", max_price="max")
    .sort_values("average_price")
    .round(2)
)
condition_summary = (
    listings.groupby("condition")["price"]
    .agg(listings="count", average_price="mean", median_price="median", min_price="min", max_price="max")
    .sort_values("average_price")
    .round(2)
)

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "manage.py").exists()), Path.cwd())
output_dir = repo_root / "data"
output_dir.mkdir(parents=True, exist_ok=True)
excel_path = output_dir / "public_api_price_report.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    listings.to_excel(writer, sheet_name="Listings", index=False)
    category_summary.to_excel(writer, sheet_name="By category")
    condition_summary.to_excel(writer, sheet_name="By condition")

print(f"Source API: {API_BASE_URL}")
print(f"Excel workbook created: {excel_path.resolve()}")
display(category_summary)
display(condition_summary)

Source API: http://127.0.0.1:8000/api/listings/
Excel workbook created: /Users/ya/Desktop/Study/Courses/INFO490/moveon/data/public_api_price_report.xlsx


,listings,average_price,median_price,min_price,max_price
category,,,,,
Home Decor,17,21.18,18.0,8.0,48.0
Appliances,3,30.00,30.0,22.0,38.0
Furniture,14,56.07,50.0,22.0,110.0
Electronics,8,92.50,75.0,20.0,200.0


,listings,average_price,median_price,min_price,max_price
condition,,,,,
NEW,2,23.50,23.5,15.0,32.0
FAIR,8,30.62,28.5,9.0,60.0
GOOD,22,42.09,35.0,8.0,120.0
LIKE_NEW,10,75.70,53.0,15.0,200.0


## Use 2: Interactive charts

**Purpose:** explore price variation by category and the number of listings in each category.

**Output:** two linked Plotly charts with a shared category-to-color mapping. Click a bar or a price point to highlight that category in both charts; click it again to clear the selection. Other categories stay visible but are dimmed.

Install `requirements-notebook.txt` before using this interactive output.

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative
from IPython.display import display

categories = sorted(listings["category"].dropna().unique())
palette = qualitative.Safe
category_colors = {
    category: palette[index % len(palette)]
    for index, category in enumerate(categories)
}
category_counts = listings.groupby("category").size()

subplot_figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.16,
    row_heights=[0.68, 0.32],
    subplot_titles=(
        "Listing price distribution",
        "Active listing count",
    ),
)
for category in categories:
    rows = listings[listings["category"] == category]
    color = category_colors[category]
    price_trace = go.Box(
        x=[category] * len(rows),
        y=rows["price"],
        name=category,
        boxpoints="all",
        jitter=0.35,
        pointpos=0,
        marker_color=color,
        line_color=color,
        customdata=rows[["title", "condition"]].to_numpy(),
        hovertemplate=(
            "<b>%{fullData.name}</b><br>"
            "Price: $%{y:.2f}<br>"
            "%{customdata[0]}<br>"
            "Condition: %{customdata[1]}"
            "<extra></extra>"
        ),
    )
    subplot_figure.add_trace(price_trace, row=1, col=1)

    count_trace = go.Bar(
        x=[category],
        y=[int(category_counts[category])],
        name=category,
        marker_color=color,
        text=[int(category_counts[category])],
        textposition="outside",
        hovertemplate=(
            "<b>%{fullData.name}</b><br>"
            "Listings: %{y}<extra></extra>"
        ),
    )
    subplot_figure.add_trace(count_trace, row=2, col=1)

subplot_figure.update_xaxes(
    categoryorder="array",
    categoryarray=categories,
    showticklabels=False,
    row=1,
    col=1,
)
subplot_figure.update_xaxes(
    categoryorder="array",
    categoryarray=categories,
    title_text="Category",
    row=2,
    col=1,
)
subplot_figure.update_yaxes(title_text="Listing price (USD)", row=1, col=1)
subplot_figure.update_yaxes(title_text="Number of listings", row=2, col=1)
figure = go.FigureWidget(
    data=subplot_figure.data,
    layout=subplot_figure.layout,
)

selection = {"category": None}

def highlight_category(category):
    selection["category"] = (
        None if selection["category"] == category else category
    )
    active = selection["category"]
    with figure.batch_update():
        for trace in figure.data:
            trace.opacity = (
                1.0 if active is None or trace.name == active else 0.2
            )

def connect_click(trace, points, selector, category):
    if points.point_inds:
        highlight_category(category)

for category, trace_index in zip(categories, range(0, len(categories) * 2, 2)):
    figure.data[trace_index].on_click(
        lambda trace, points, selector, category=category:
            connect_click(trace, points, selector, category)
    )
    figure.data[trace_index + 1].on_click(
        lambda trace, points, selector, category=category:
            connect_click(trace, points, selector, category)
    )

figure.update_layout(
    title="Listing prices and counts by category",
    height=max(700, 150 * len(categories)),
    showlegend=False,
    hovermode="closest",
    margin={"t": 100, "b": 50},
)
display(figure)

FigureWidget({
    'data': [{'boxpoints': 'all',
              'customdata': array([['Compact Microwave', 'LIKE_NEW'],
                                   ['Countertop Microwave', 'GOOD'],
                                   ['Microwave', 'GOOD']], dtype=object),
              'hovertemplate': ('<b>%{fullData.name}</b><br>Pri' ... '{customdata[1]}<extra></extra>'),
              'jitter': 0.35,
              'line': {'color': 'rgb(136, 204, 238)'},
              'marker': {'color': 'rgb(136, 204, 238)'},
              'name': 'Appliances',
              'pointpos': 0,
              'type': 'box',
              'uid': '5eebc277-4aec-4fa2-958c-5a0aa80da82b',
              'x': [Appliances, Appliances, Appliances],
              'xaxis': 'x',
              'y': {'bdata': 'AAAAAAAAQ0AAAAAAAAA2QAAAAAAAAD5A', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': '<b>%{fullData.name}</b><br>Listings: %{y}<extra></extra>',
              'marker': {'color': 'rgb(136, 204, 23

## Use 3: Simple price prediction experiment

**Purpose:** demonstrate a basic regression workflow. Category and condition are features; listing price is the target.

**Output:** test-set MAE for a median-price baseline and one-hot encoded Ridge regression, plus actual-versus-predicted rows. The split is reproducible.

**Limit:** these are asking prices, not sale prices. A small or unbalanced API sample makes the error estimate unstable; this is a technical demonstration rather than a reliable valuation tool.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

feature_columns = ["category", "condition"]
if len(listings) < 8:
    raise ValueError("At least 8 listings are needed for this illustrative train/test split; the API sample is too small.")

X = listings[feature_columns].copy()
y = listings["price"].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

preprocess = ColumnTransformer([
    ("categorical", make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(handle_unknown="ignore"),
    ), feature_columns),
])
model = make_pipeline(preprocess, Ridge(alpha=1.0))
model.fit(X_train, y_train)
model_predictions = model.predict(X_test)

scores = pd.DataFrame({
    "method": ["Training-set median baseline", "Category + condition Ridge model"],
    "test_MAE_USD": [
        mean_absolute_error(y_test, baseline_predictions),
        mean_absolute_error(y_test, model_predictions),
    ],
}).round(2)
comparison = pd.DataFrame({
    "category": X_test["category"].to_numpy(),
    "condition": X_test["condition"].to_numpy(),
    "actual_price": y_test.to_numpy(),
    "predicted_price": model_predictions.round(2),
})
print(f"API: {API_BASE_URL} | rows: {len(listings)} | train: {len(X_train)} | test: {len(X_test)}")
display(scores)
display(comparison.head(10))

API: http://127.0.0.1:8000/api/listings/ | rows: 42 | train: 31 | test: 11


,method,test_MAE_USD
0,Training-set median baseline,24.55
1,Category + condition Ridge model,18.26


,category,condition,actual_price,predicted_price
0,Home Decor,FAIR,9.0,4.93
1,Electronics,GOOD,40.0,73.32
2,Furniture,GOOD,70.0,52.92
3,Home Decor,LIKE_NEW,28.0,51.86
4,Furniture,GOOD,40.0,52.92
5,Electronics,GOOD,120.0,73.32
6,Home Decor,GOOD,20.0,15.51
7,Home Decor,FAIR,18.0,4.93
8,Electronics,GOOD,85.0,73.32
9,Appliances,LIKE_NEW,38.0,68.15
